<a href="https://colab.research.google.com/github/RajeevKambhatla/longevity-drug-repurposing/blob/main/04_llm_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q anthropic biopython pydantic tenacity
from google.colab import drive, userdata
drive.mount('/content/drive')

import json, os, time, pandas as pd
BASE = '/content/drive/MyDrive/longevity_project'
os.makedirs(f'{BASE}/data/processed/abstracts', exist_ok=True)
os.makedirs(f'{BASE}/results/reviews', exist_ok=True)

scores = pd.read_csv(f'{BASE}/results/senescence_baseline_pilot.csv')
top = scores.head(5)
print(top[['name', 'score', 'rank']])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 70.2 MB/s eta 0:00:00
Mounted at /content/drive
            name     score  rank
0      st-014366  1.761635     1
1  naftidrofuryl  1.694840     2
2      practolol  1.634028     3
3     valdecoxib  1.587852     4
4     emd-386088  1.573991     5


In [2]:
from Bio import Entrez
Entrez.email = 'your.email@example.com'   # NCBI asks for an email; use yours

def fetch_abstracts(drug, n=8):
    name = drug.replace('-', ' ')
    q = (f'"{name}"[Title/Abstract] AND (senescence[Title/Abstract] OR senescent[Title/Abstract] '
         f'OR SASP[Title/Abstract] OR aging[Title/Abstract])')
    ids = Entrez.read(Entrez.esearch(db='pubmed', term=q, retmax=n, sort='relevance'))['IdList']
    if not ids:
        return []
    recs = Entrez.read(Entrez.efetch(db='pubmed', id=','.join(ids), retmode='xml'))
    out = []
    for art in recs['PubmedArticle']:
        a = art['MedlineCitation']['Article']
        abstract = ' '.join(str(x) for x in a.get('Abstract', {}).get('AbstractText', []))
        out.append({'pmid': str(art['MedlineCitation']['PMID']), 'title': str(a['ArticleTitle']), 'abstract': abstract})
    return out

In [3]:
for drug in top['name']:
    arts = fetch_abstracts(drug)
    json.dump(arts, open(f'{BASE}/data/processed/abstracts/{drug}.json', 'w'))
    print(drug, len(arts), 'articles')
    time.sleep(0.5)

st-014366 0 articles
naftidrofuryl 3 articles
practolol 0 articles
valdecoxib 3 articles
emd-386088 0 articles


In [4]:
from pydantic import BaseModel
from typing import Literal

class Review(BaseModel):
    drug: str
    believability: Literal['High', 'Medium', 'Low']
    likely_mechanism: str
    evidence_summary: str
    concerns: str
    key_pmids: list[str]

In [5]:
import anthropic
from tenacity import retry, stop_after_attempt, wait_exponential
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

SYSTEM = """You are a skeptical aging-biology reviewer judging whether an existing drug might reduce cellular senescence.
Base your judgment only on the articles provided and the drug's score. Be skeptical: a high score alone is weak evidence,
and missing literature means believability is Low. Note concerns such as toxicity or signs the drug simply stresses cells.
Respond with only a JSON object with these keys: drug, believability (High, Medium, or Low), likely_mechanism,
evidence_summary, concerns, key_pmids (a list, using only PMIDs from the articles provided)."""

@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=2, max=20))
def review(drug, score, rank, articles):
    text = '\n\n'.join(f"PMID {a['pmid']}: {a['title']}\n{a['abstract']}" for a in articles) or 'No articles found.'
    prompt = (f"Drug: {drug}\nSenescence score in VCAP cells: {score:.2f} (rank {rank} of 100; "
              f"higher means senescence genes were pushed down more)\n\nArticles:\n{text}")
    msg = client.messages.create(model='claude-sonnet-5', max_tokens=1000, system=SYSTEM,
                                 messages=[{'role': 'user', 'content': prompt}])
    raw = msg.content[0].text.strip().removeprefix('```json').removesuffix('```').strip()
    return Review.model_validate_json(raw)

In [6]:
row = top.iloc[0]
arts = json.load(open(f"{BASE}/data/processed/abstracts/{row['name']}.json"))
r = review(row['name'], row['score'], row['rank'], arts)
print(r.model_dump_json(indent=2))

{
  "drug": "st-014366",
  "believability": "Low",
  "likely_mechanism": "Unknown - no supporting literature available to infer mechanism",
  "evidence_summary": "The only evidence provided is a single computational senescence score (1.76, ranked 1 of 100) in VCAP cells, derived from gene expression changes. No published articles, preclinical studies, or mechanistic data were found to corroborate this finding or explain how st-014366 might affect senescence pathways.",
  "concerns": "No independent literature exists to confirm this is a genuine anti-senescence effect rather than a transcriptional artifact, off-target toxicity, or cell-stress response that happens to downregulate senescence-associated genes without improving cellular health. A top rank in a single assay/cell line is weak evidence on its own; it could reflect general cytotoxicity, stress-induced suppression of senescence markers, or assay-specific noise. Without orthogonal validation (SA-β-gal, p16/p21 protein levels, SA

In [7]:
for _, row in top.iterrows():
    path = f"{BASE}/results/reviews/{row['name']}.json"
    if os.path.exists(path):
        continue
    arts = json.load(open(f"{BASE}/data/processed/abstracts/{row['name']}.json"))
    r = review(row['name'], row['score'], row['rank'], arts)
    open(path, 'w').write(r.model_dump_json(indent=2))
    print(row['name'], '->', r.believability)

st-014366 -> Low
naftidrofuryl -> Low
practolol -> Low
valdecoxib -> Low
emd-386088 -> Low


In [8]:
for drug in top['name']:
    r = json.load(open(f'{BASE}/results/reviews/{drug}.json'))
    real = {a['pmid'] for a in json.load(open(f'{BASE}/data/processed/abstracts/{drug}.json'))}
    bad = [p for p in r['key_pmids'] if p not in real]
    print(drug, r['believability'], 'invented PMIDs:' if bad else 'citations OK', bad or '')

st-014366 Low citations OK 
naftidrofuryl Low citations OK 
practolol Low citations OK 
valdecoxib Low citations OK 
emd-386088 Low citations OK 
